### Copy Number Variation (CNV) Analysis Pipeline

#### Overview
This notebook performs a comprehensive statistical analysis of copy number variations in head and neck cancer patients, stratified by HPV status. The analysis identifies significantly amplified and deleted genes using GISTIC-like scoring methods with dual statistical validation approaches (binomial and empirical tests).

#### Methodology

##### 1. Data Preprocessing
- **Input Data**: TCGA gene-level CNV data from the Nulton cohort
- **CNV Thresholds**:
  - **High-level amplification**: Copy number > 4
  - **High-level deletion**: Copy number < 1
  - **Outlier cap**: CNV values capped at 7 to prevent outliers from skewing results
- **Gene Annotation**: Gene positions and chromosome coverage calculated from GRCh38 assembly
- **Cohort Stratification**: HPV-positive vs HPV-negative patients

##### 2. CNV Event Counting and Scoring
For each gene, we calculate:
- **Amplification/Deletion count**: Number of samples with high-level CNV events
- **CNV sum**: For amplifications, sum of log2(CNV - 1) across samples; for deletions, sum of deletion penalties (-1 for CNV=1, -2 for CNV<1)
- **CNV prevalence**: Average CNV score across all samples
- **Frequency percentage**: Percentage of samples with high-level CNV events
- **GISTIC score**: Composite score combining CNV intensity (prevalence) and frequency

##### 3. Statistical Significance Testing

###### A. Binomial Test (Parametric Approach)
Tests whether observed CNV events in a gene exceed what would be expected by random genomic instability.

**Null Hypothesis**: CNV events are randomly distributed across genes proportional to the genome-wide background CNV rate.

###### B. Empirical/Permutation Test (Non-parametric Approach)
Generates a null distribution through permutation testing to validate binomial test results.

**Approach**: Randomly sample CNV values (with replacement) 1,000 times to create null distributions.

##### 4. Multiple Testing Correction
- **Method**: Benjamini-Hochberg False Discovery Rate (FDR) correction
- **Applied to**: Both binomial q-values and empirical q-values
- **Significance threshold**: Adjusted q-value < 0.05

##### 5. Gene Filtering Criteria
Genes are considered significant if they meet both criteria:
- Binomial q-value < 0.05
- Empirical q-value < 0.05

#### Output
- Gene-level CNV statistics for HPV+ and HPV- cohorts
- Separate results for amplifications and deletions
- Filtered lists of significantly altered genes passing both statistical tests
- Chromosome coverage information for each gene

### Mathematical Formulas for Statistical Tests

#### Binomial Test Formula

##### Test Setup for CNV Events
For each gene $g$, we test whether the observed number of high-level CNV events exceeds random expectation based on genome-wide CNV instability.

**Parameters:**
- $x$ = observed number of samples with high-level CNV events in gene $g$ (CNV > 4 for amplifications or CNV < 1 for deletions)
- $n$ = total number of samples (patients) in the cohort
- $p_{\text{null}}$ = genome-wide background CNV rate

##### Background CNV Rate Calculation
The null probability is calculated as the proportion of all CNV observations that exceed the high-level threshold:

**For amplifications:**
$$p_{\text{null}} = \frac{\sum_{i=1}^{N} \mathbb{1}(\text{CNV}_i > 4)}{N}$$

**For deletions:**
$$p_{\text{null}} = \frac{\sum_{i=1}^{N} \mathbb{1}(\text{CNV}_i < 1)}{N}$$

where:
- $N$ = total number of CNV observations across all genes and samples
- $\mathbb{1}(\cdot)$ = indicator function (1 if condition is true, 0 otherwise)

##### Binomial Test Statistic
Under the null hypothesis (CNV events randomly distributed based on genomic instability), the number of high-level CNV events in gene $g$ follows a binomial distribution:

$$x \sim \text{Binomial}(n, p_{\text{null}})$$

The probability of observing $x$ or more CNV events in gene $g$ by chance is:

$$P\text{-value}_{\text{binomial}} = P(X \geq x) = \sum_{k=x}^{n} \binom{n}{k} p_{\text{null}}^k (1-p_{\text{null}})^{n-k}$$

where $X \sim \text{Binomial}(n, p_{\text{null}})$.

**Alternative form (right-tailed test):**

$$P\text{-value}_{\text{binomial}} = 1 - \sum_{k=0}^{x-1} \binom{n}{k} p_{\text{null}}^k (1-p_{\text{null}})^{n-k}$$

This GISTIC-like approach tests each gene against the observed cohort-specific genomic instability, where $p_{\text{null}}$ represents the baseline rate at which high-level CNV events occur across the entire genome.

---

#### Empirical/Permutation Test Formula

##### Test Setup
To validate the binomial test results, we generate an empirical null distribution through permutation testing.

**Parameters:**
- $M$ = number of permutations (1,000 in this analysis)
- $n$ = sample size (number of samples/patients in cohort)
- $x_{\text{obs}}$ = observed number of samples with high-level CNV events for the gene

##### Permutation Procedure
For each permutation $m$:
1. Randomly sample $n$ CNV values (with replacement) from the pooled set of all CNV values across all genes and samples
2. Count how many sampled values exceed the threshold (CNV > 4 for amplifications, CNV < 1 for deletions)
3. Record this count as $\text{count}_m$

This creates a null distribution of what we would expect to see by chance given the overall CNV landscape of the cohort.

##### Empirical P-value Calculation
After $M$ permutations, the empirical p-value is calculated as:

$$p_{\text{empirical}} = \frac{1 + \sum_{m=1}^{M} \mathbb{1}(\text{count}_m \geq x_{\text{obs}})}{M + 1}$$

where:
- $x_{\text{obs}}$ = observed number of samples with high-level CNV events for the gene
- $\text{count}_m$ = simulated count of high-level CNV events in permutation $m$
- $\mathbb{1}(\cdot)$ = indicator function (1 if condition is true, 0 otherwise)
- The "+1" in numerator and denominator prevents p-values of exactly 0

**Interpretation:** This represents the proportion of permutations where the simulated count equals or exceeds the observed count, providing a data-driven assessment of significance.

---

#### Multiple Testing Correction

##### Benjamini-Hochberg FDR Correction
Both binomial p-values and empirical p-values are corrected for multiple testing using the Benjamini-Hochberg procedure to control the False Discovery Rate.

**Procedure:**
1. Order p-values: $p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(G)}$
2. For each ordered p-value at rank $i$, calculate the adjusted q-value:

$$q_{(i)} = \min\left(1, \min_{j \geq i} \left\{ \frac{G \cdot p_{(j)}}{j} \right\} \right)$$

3. Report genes with adjusted q-value $\leq \alpha$ (typically $\alpha = 0.05$)

where $G$ is the total number of genes tested.

---

#### GISTIC-like Composite Score

The GISTIC score balances CNV intensity (amplitude) with cohort prevalence (frequency).

##### For Amplifications

**CNV Sum (Amplitude):**
$$\text{CNV Sum}_g = \sum_{s=1}^{n} \begin{cases} 
\log_2(\text{CNV}_{g,s}) - 1 & \text{if CNV}_{g,s} > 1 \\
0 & \text{otherwise}
\end{cases}$$

where CNV values are capped at 7 to prevent outliers: if $\text{CNV}_{g,s} > 7$, use $\text{CNV}_{g,s} = 7$.

**CNV Prevalence:**
$$\text{CNV Prevalence}_g = \frac{\text{CNV Sum}_g}{n}$$

**Frequency Percentage:**
$$\text{Frequency \%}_g = \frac{\text{Count}(\text{CNV}_{g,s} > 4)}{n} \times 100$$

**GISTIC Score:**
$$\text{GISTIC Score}_g = \text{CNV Prevalence}_g \times \frac{\text{Frequency \%}_g}{100}$$

### For Deletions

**Deletion Sum (Amplitude):**
$$\text{Deletion Sum}_g = \sum_{s=1}^{n} \begin{cases} 
-2 & \text{if CNV}_{g,s} < 1 \\
-1 & \text{if CNV}_{g,s} = 1 \\
0 & \text{otherwise}
\end{cases}$$

**Deletion Prevalence (absolute value):**
$$\text{Deletion Prevalence}_g = \frac{|\text{Deletion Sum}_g|}{n}$$

**Frequency Percentage:**
$$\text{Frequency \%}_g = \frac{\text{Count}(\text{CNV}_{g,s} < 1)}{n} \times 100$$

**GISTIC Score:**
$$\text{GISTIC Score}_g = \text{Deletion Prevalence}_g \times \frac{\text{Frequency \%}_g}{100}$$

## Analysis Pipeline

### Libraries

In [ ]:
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import numpy as np
import math
from math import log10

import random

### Data

In [ ]:
### Sample sheet
file_path = 'Data/TCGA/Gene level CNV/gdc_sample_sheet.2025-11-03.tsv'
sample_sheet = pd.read_csv(file_path, sep = '\t')

In [ ]:
sample_sheet

In [ ]:
cases = []
for val in sample_sheet['Case ID']:
    temp = [x.strip() for x in val.split(',')]
    for val2 in temp:
        cases.append(val2)
    if len(set(temp)) >=1:
        print(val)
#print(len(set(cases)))

In [ ]:
for case in list(set(cases)):
    print(case)
print("number of unique case ids from sample sheet: ", len(set(cases)))

In [ ]:
print(f"Number of samples: {len(sample_sheet['Sample ID'].unique())}")
print(f"Number of Project: {len(sample_sheet['Project ID'].unique())}")
print(f"Number of patients: {len(set(cases))}")
print(f"Number of unique cases: {len(set(cases))}")
print(f"Number of patients: {len(set(cases))}")

In [ ]:
def num_overlap(list1,list2):
    # Convert lists to sets
    set1 = set(list1)
    set2 = set(list2)

    # Find the intersection of the two sets
    overlap = set1.intersection(set2)

    # Get the number of overlapping values
    num_overlap = len(overlap)
    return num_overlap

### Seperate on HPV status

Split samples based on HPV+ or HPV- to identify drug repurposing cadndidates based on this stratification

To be modified with Travis input of HPV status patients. For now basing on CDKN2A status

In [ ]:
# Specify the folder path
folder_path = 'Data/TCGA/Gene level CNV'

# Get the list of all files in the specified folder
folders = os.listdir(folder_path)

files_to_work = []
for folder in folders:
    if ('clinical' not in folder and 'biospecimen' not in folder):
        try:
            current_folder = os.listdir(f'{folder_path}/{folder}')
            for file in current_folder:
                if '.tsv' in file:
                    files_to_work.append(f'{folder_path}/{folder}/{file}')
        except:
            continue

In [ ]:
len(files_to_work)

In [ ]:
files_to_work

In [ ]:
HPV_positive_df = pd.read_csv('Results/HPV results/HPV positive patients.csv')
HPV_positive_cases = [x for x in HPV_positive_df['TCGA CASE ID']]
HPV_positive_cases = list(set(HPV_positive_cases))
HPV_negative_df = pd.read_csv('Results/HPV results/HPV negative patients.csv')
HPV_negative_cases = [x for x in HPV_negative_df['TCGA CASE ID']]
HPV_negative_cases = list(set(HPV_negative_cases))

In [ ]:
print((len(HPV_positive_df)))
print((len(HPV_negative_cases)))

In [ ]:
accounted_for = []
HPV_positive_files = []
HPV_negative_files = []
none_file = []
for file in files_to_work:
    temp = file.split('/')[3]
    #print(temp)
    #print(len(sample_sheet[sample_sheet['File ID']==temp]['Case ID'].values))
    file_cases = list(set(sample_sheet[sample_sheet['File ID']==temp]['Case ID'].values))
    #print(file_cases)
    file_cases = file_cases[0].split(',')
    for file_case in file_cases:
        file_case = file_case.replace(' ','')
        if file_case in HPV_positive_cases:
            accounted_for.append(file_case)
            #print(sample_sheet[sample_sheet['Case ID']==file_case]['File ID'].values)
            HPV_positive_files.append(file)
        else:
            HPV_negative_files.append(file)
        
#print(HPV_positive_files)
# HPV_positive_files = list(set(HPV_positive_files))
HPV_positive_files = list(set(HPV_positive_files))
print(len(HPV_positive_files))
#print(len(HPV_negative_files))

## check if all files are accounted for
none_file = []
for file in files_to_work:
    if (file not in HPV_positive_files) and (file not in HPV_negative_files):
        none_file.append(file)

print(len(set(HPV_negative_files)))
print(len(none_file))
print(len(set(accounted_for)))

In [ ]:
### double check the cases of these files
temp_HPV_positive_files = [file.split('/')[3] for file in HPV_positive_files]
temp_HPV_negative_files = [file.split('/')[3] for file in HPV_negative_files]

sample_sheet_hpv_positive = sample_sheet[sample_sheet['File ID'].isin(temp_HPV_positive_files)]
sample_sheet_hpv_negative = sample_sheet[sample_sheet['File ID'].isin(temp_HPV_negative_files)]

### check that correct case ids are there
temp_hpv_positive_case_ids = list(sample_sheet_hpv_positive['Case ID'].unique())
temp_hpv_negative_case_ids = list(sample_sheet_hpv_negative['Case ID'].unique())

#temp_hpv_positive_case_ids = [x.split(',')[0].replace(' ','') for x in temp_hpv_positive_case_ids]
hpv_positive_cleaned_case_ids = []
for i, case_id in enumerate(temp_hpv_positive_case_ids):
    case_id_split = case_id.split(',')
    case_id_split = [x.strip() for x in case_id_split]
    #print(case_id_split)
    if len(case_id_split) > 1:
        # Mark this index for removal and add individual case IDs to new list
        for case_ids in case_id_split:
            hpv_positive_cleaned_case_ids.append(case_ids)
    else:
        hpv_positive_cleaned_case_ids.append(case_id_split[0])

hpv_positive_cleaned_case_ids = list(set(hpv_positive_cleaned_case_ids))

hpv_negative_cleaned_case_ids = []
for i, case_id in enumerate(temp_hpv_negative_case_ids):
    case_id_split = case_id.split(',')
    case_id_split = [x.strip() for x in case_id_split]
    #print(case_id_split)
    if len(case_id_split) > 1:
        # Mark this index for removal and add individual case IDs to new list
        for case_ids in case_id_split:
            hpv_negative_cleaned_case_ids.append(case_ids)
    else:
        hpv_negative_cleaned_case_ids.append(case_id_split[0])

hpv_negative_cleaned_case_ids = list(set(hpv_negative_cleaned_case_ids))

print(f"Number of HPV positive cases: {len(hpv_positive_cleaned_case_ids)}")
print(f"Number of HPV negative cases: {len(hpv_negative_cleaned_case_ids)}")

for positive_case_id in hpv_positive_cleaned_case_ids:
    if positive_case_id not in HPV_positive_cases:
        print(f"Missing HPV positive case ID: {positive_case_id}")

for negative_case_id in hpv_negative_cleaned_case_ids:
    if negative_case_id not in HPV_negative_cases:
        print(f"Missing HPV negative case ID: {negative_case_id}")

In [ ]:
print(len(HPV_negative_files))
print(len(HPV_positive_files))
len(HPV_negative_cases)

In [ ]:
print(len(set(HPV_negative_files)))
print(len(set(HPV_positive_files)))
print(len(set(HPV_negative_cases)))

In [ ]:
### only keep unique files and cases
HPV_negative_files = list(set(HPV_negative_files))
HPV_positive_files = list(set(HPV_positive_files))

In [ ]:
overlap = set(HPV_positive_files) & set(HPV_negative_files)
print(f"Files in both groups: {len(overlap)}")

In [ ]:
HPV_negative_files

In [ ]:
print(len(HPV_negative_files))
print(len(HPV_positive_files))
print(len(HPV_negative_cases))
print(len(HPV_positive_cases))

In [ ]:
metadata = pd.read_csv('Data/TCGA/Gene level CNV/gdc_sample_sheet.2025-11-03.tsv', sep='\t')

In [ ]:
metadata

## Drug repurposing

In [ ]:
# temp = pd.read_csv(files_to_work[0], sep = '\t')
# temp = temp[temp['gene_id'].notna()]
# temp[~temp['copy_number'].isna()]
# # ### view where in table gene id is repeated
# # temp[temp.duplicated(subset=['gene_name'], keep=False)]
# # len(temp['gene_id'].unique())
# temp[~temp['copy_number'].isna()]['gene_name'].nunique()

In [ ]:
### Check if all files accounted for
print(len(set(HPV_negative_files)))
print(len(set(HPV_positive_files)))
print(len(set(HPV_positive_files).union(set(HPV_negative_files))))

In [ ]:
len(set(files_to_work))

In [ ]:
### by file each gene copy number is saved
file_dict_gene_dict = {}
for file in tqdm(files_to_work):
    tempDict = {}
    temp = pd.read_csv(file, sep='\t')
    for i,gene in enumerate(temp['gene_id']):
        tempDict[gene] = temp.iloc[i].loc['copy_number']
    file_dict_gene_dict[file] = tempDict

In [ ]:
### gene id to name translation dict
gene_id_translation_dict = {}
for file in tqdm(files_to_work):
    temp = pd.read_csv(file, sep='\t')
    for i,gene in enumerate(temp['gene_id']):
        if gene in gene_id_translation_dict:
            continue
        else:
            gene_id_translation_dict[gene] = temp.iloc[i].loc['gene_name']

In [ ]:
len(gene_id_translation_dict)

In [ ]:
### graph cnv values to note the cutoff of 7 for outlier CNV values
gene_cnv_values = []
for file in tqdm(files_to_work):
    for gene in file_dict_gene_dict[file]:
        if not math.isnan(file_dict_gene_dict[file][gene]):
            gene_cnv_values.append(file_dict_gene_dict[file][gene])


In [ ]:
len(gene_cnv_values)

In [ ]:
pd.Series(gene_cnv_values)

In [ ]:
### create a counts table for gene CNV values, make a df for easy viewing
gene_cnv_counts = pd.DataFrame(pd.Series(gene_cnv_values).value_counts()).reset_index()
gene_cnv_counts.columns = ['CNV Value', 'Frequency']

In [ ]:
gene_cnv_counts.sort_values(by = 'Frequency', ascending=True).head(15)

In [ ]:
#### plot the gene_cnv_counts to see frequency of CNV values
outlier_threshold = 7
plot_gene_cnv_counts = gene_cnv_counts.sort_values(by = 'CNV Value', ascending=True)
plt.figure(figsize=(10,6))
plt.bar(plot_gene_cnv_counts['CNV Value'], plot_gene_cnv_counts['Frequency'])
plt.axvline(x=outlier_threshold, color='r', linestyle='--', label=f'Outlier threshold: {outlier_threshold}')
plt.title('Distribution of Gene CNV Values')
plt.xlabel('CNV Value')
plt.yscale('log')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
def gistic_scoring_mimic_amplification(files, file_dict_gene_dict, gene_id_translation_dict):
    """
    Mimics the GISTIC scoring for amplification events. This version will create a null based on the shuffled amplification values. 
    Also works to calculate the percentage of a chromosome the mutation covers
    
    Parameters:
    files (list): List of file paths to process.
    file_dict_gene_dict (dict): Dictionary mapping file names to gene dictionaries. contain the copy number values of each gene
    with >4 indicating high level amplification.

    
    Returns:
    dict: A dataframe with scores for each gene.
    1. 'gene': Gene identifier.
    2. 'gene_name': Human-readable gene name.
    3. 'Sample_size': Total number of samples (files) processed.
    4. 'amplification_count': Count of amplification events for the gene.
    5.  Amplification_sum (i.e. total log2(amplification CNV) across samples): the sum of log2 values of amplification CNV across samples. (-1 to adjust for diploid cells, so only amplification values and intensity are counted for)
        This is similar to GISTIC's approach of assessing amplification events.
    6.  Amplification_sum_prevalence (i.e. average log2(amplification CNV) across samples): the average log2 value of amplification CNV across samples. (amplification_sum / sample_count)
    7. frequency_percentage: Percentage of samples with amplification for the gene. (number of files where gene is amplfied significantly (CNV > 4) / total number of files * 100)
    8. 'gistic_score': GISTIC-like score for the gene based on average amplification value.
    9. 'p_value': P-value from the binomial test for amplification.
    10. 'q_value': Adjusted p-value for multiple testing (FDR).
    11. 'significant': Boolean indicating if the gene is significantly amplified.
    12. 'empirical_p_value': Empirical p-value based on the null distribution.
    13. 'empirical_q_value': Adjusted empirical p-value for multiple testing (FDR).
    14. 'empirical_significant': Boolean indicating if the gene is significantly amplified based on the empirical p-value.
    15. 'start': Start position of the gene on the chromosome.
    16. 'end': End position of the gene on the chromosome.
    17. 'length': Length of the gene in base pairs (end - start + 1).
    18. 'chromosome': Chromosome on which the gene is located.
    19. 'coverage_percentage': Percentage of the chromosome that the gene covers.
    """



    ########################### creation of the calculations for the amplification effects #################################
    files = list(set(files)) # Ensure files is a list of unique file paths
    sample_count = len(files)
   
    # Step 1: Count amplification events and accumulate values
    ### Initialize dictionaries to hold amplification counts and values
    ### amplification_dict will hold the count of amplification events for each gene

    amplification_dict = {}
    amp_values_sum = {}

    ### initialize amplification_dict with 0 for all genes
    all_genes = set()
    for f in files:
        all_genes.update(file_dict_gene_dict[f].keys())
    amplification_dict = {gene: 0 for gene in all_genes}
    amp_values_sum = {gene: 0 for gene in all_genes}

    ### amp_values_sum will hold the sum of values for each gene
    cnv_values = []
    for file in tqdm(files, desc = "Counting amplification events and accumulating values"):
        gene_dict = file_dict_gene_dict.get(file, {})
        for gene, value in gene_dict.items():
            ### skip if values is none
            # check if the value is NaN, 'NA', 'nan', or not a number
            ### this is to ensure that we only consider valid numerical values for amplification
            if pd.isnull(value) or value == 'NA' or value == 'nan' or not isinstance(value, (int, float, np.integer, np.floating)):
                cnv_values.append(2)  # Assuming a default value of 2 for missing or invalid CNV values, which indicates no amplification
                amp_values_sum[gene] = amp_values_sum.get(gene, 0) + 0
                continue
            cnv_values.append(value)
            # Mimicking GISTIC scoring for amplification
            
            ### this section of code works to accumulate the amplification values for each gene
            ### if the value is less than or equal to 1, it indicates no amplification
            ### and we do not accumulate the value for that gene
            ### if the value is greater than 1, we accumulate the log2 of the value
            ### this is similar to GISTIC's approach of assessing amplification events
            ### capping the value at 7 to prevent outliers from skewing the results
            ### this is similar to GISTIC's approach of assessing amplification events
            ### log2 transformation is used to account for when values are 2 as in normal diploid cells
            ### and to reduce the impact of outliers on the results
            ### Note: GISTIC uses log2 transformation to reduce the impact of outliers
            if value <=1:  # Assuming values <=1 indicate no amplification
                amp_values_sum[gene] = amp_values_sum.get(gene, 0) + 0
                ### CNV values are only integers so we are okay skipping between 1 and 2, 
                # which would indicate a deletion or normal diploid state, and only counting amplifications
            else:
                ### accumulate the values for each gene
                if value > 7: # prevent outliers from skewing the results
                    value = 7
                ### accumulate the sum of values for each gene
                amp_values_sum[gene] = amp_values_sum.get(gene, 0) + math.log2(value)-1  # -1 to adjust for diploid cells
            
            ### seperate check for high-level amplification
            ### assuming values >4 indicate high-level amplification
            ### this is similar to GISTIC's approach of assessing amplification events
            if value > 4:  # Assuming values >4 indicate high-level amplification
                ### accumulate the counts of amplification events for each gene
                amplification_dict[gene] = amplification_dict.get(gene, 0) + 1

    # Step 2: Binomial test per gene
    ### Perform binomial test for each gene
    # Note: The binomial test checks if the observed count is significantly greater than expected under the null hypothesis
    # where the null hypothesis is that the probability of amplification is p_null.
    # The alternative hypothesis is that the observed count is significantly greater than expected.
    # This is similar to GISTIC's approach of assessing significance of amplification events.
    
    ### set p_null ot the background prevalence of amplification events across all genes
    p_null = np.sum(np.array(cnv_values) > 4) / len(cnv_values)
    genes = list(amplification_dict.keys())
    counts = [amplification_dict[g] for g in genes]
    p_values = []
    ### count is the number of significant/high-level amplification events observed for each gene
    # sample_count = len(files)  # Total number of samples (files)
    ### The p_null is set the background prevalence of amplification events across all genes
    # alternative='greater' indicates we are testing if the observed count is significantly greater than expected
    for count in counts:
        result = binomtest(k=count, n=sample_count, p=p_null, alternative='greater')
        p_values.append(result.pvalue)

    ### Calculate a null distribution based on the shuffled amplification values
    # Create a null distribution by permuting the CNV values
    ### This will help in calculating empirical p-values
    ### Initialize a dictionary to hold the null distributions for each gene
    ### This will hold the counts of amplification events for each gene in the null distribution

    # shuffle the CNV values to create a null distribution
    np.random.seed(42)  # For reproducibility
    ### Initialize a dictionary to hold the null distributions for each gene
    null_distributions = {g: [] for g in genes}
    permutation_count = 1000  # Number of permutations to create a null distribution
    ### create quick permuted CNV matrix
    ### This will hold the CNV values for each gene across all samples
    permuted_cnv_values = np.random.permutation(cnv_values)
    ### Calculate the null distributions for each gene
    for gene in tqdm(genes, desc='Calculating null distributions without permuting every time'):
        ### This will help in calculating empirical p-values
        ### Null distribution counts the number of times the random value of the gene is a significant mutation >4 for CNV
        null_distributions[gene] = [np.sum(np.random.choice(permuted_cnv_values, size = sample_count)>4) for _ in range(permutation_count)]

    ### calculate empirical p-values based on the null distribution
    p_empirical_values = []
    for gene in genes:
        observed_count = amplification_dict.get(gene, 0)
        null_counts = null_distributions[gene]
        # Calculate the empirical p-value as the proportion of null counts greater than or equal to the observed count
        ### works by counting how many times the null distribution has a count greater than or equal to the observed count
        ### The empirical p-value is calculated as the proportion of null counts that are greater than or equal to the observed count
        ### this is divided by the total number of permutations plus one to account for the observed count itself
        ### This gives the empirical p-value for the gene
        ### Note: This is a more efficient way to calculate the empirical p-value
        ### if the p_empirical is less than the p_null, then the gene is considered significantly amplified
        ### this is similar to the binomial test, but uses the null distribution to calculate the p-value
        ### This is because the null distribution is based on the shuffled CNV values, which is more representative of the data
        p_empirical = (np.sum(np.array(null_counts)>= observed_count)+1) / (permutation_count +1)
        p_empirical_values.append(p_empirical)
    

    # Step 3: Adjust p-values for multiple testing
    rejected, q_values, _, _ = multipletests(p_values, method='fdr_bh')
    # Adjusted q-values are the p-values adjusted for multiple testing
    rejected_empirical, q_values_empirical, _, _ = multipletests(p_empirical_values, method='fdr_bh')

    # Adjusted q-values are the p-values adjusted for multiple testing
    # Note: `rejected` indicates which hypotheses are rejected (i.e., significant)
    ### my own double check
    threshold = 0.05
    my_own_rejects = [True if q < threshold else False for q in q_values]
    my_own_rejects_empirical = [True if q < threshold else False for q in q_values_empirical]

    ### collect the prevalence of the genes high amplification across the entire cohort of samples
    frequency_percentage = [100 * amplification_dict.get(gene, 0) / sample_count for gene in genes]

    # Note: GISTIC scoring is typically more complex and involves additional steps like segmenting the data,
    # but here we mimic it by averaging the amplification values across samples.
    # This is a simplified version and may not capture all nuances of GISTIC scoring.
    # GISTIC scores are calculated as the average amplification value across samples for each gene.
    # attempt at gistic scoring

    gene_amplification_sum = [amp_values_sum.get(gene, 0) for gene in genes] # This is the sum of log2 values of CNV values across samples for each gene,
    #-1 to adjust for diploid cells and essentially only count amplifications and not a negative value or incorrect reading due to deletion
    amp_values_prevalance = [(amp_values_sum.get(gene, 0) / sample_count) for gene in genes]
    # This is the average log2 value of CNV values across samples for each gene

    # Note: This is a simplified version of GISTIC scoring, which typically involves more complex calculations.
    gistic_scores = list(np.array(amp_values_prevalance) * (np.array(frequency_percentage)/100)) # This is a simplified version of GISTIC scoring
    ### balances strength * prevalence

    ### Translate gene IDs to gene names
    # This will replace gene IDs with human-readable gene names using the translation dictionary
    gene_names = [gene_id_translation_dict.get(gene, "NA") for gene in genes]

    ### essentiall the observed counts of amplification events for each gene
    # counts = [amplification_dict[g] for g in genes]

    # Step 4: Compile results
    results = []
    for gene, gene_name, count, amp_sum, pval, pval_empirical, qval, qval_empirical, reject, empirical_reject, freq, amp_sum_prev, gistic in zip(genes, gene_names, counts, gene_amplification_sum,
                                                                                                                                                 p_values, p_empirical_values, q_values, 
                                                                                                                                                 q_values_empirical, my_own_rejects, my_own_rejects_empirical, 
                                                                                                                                                 frequency_percentage, amp_values_prevalance, gistic_scores):
        results.append({
            'gene': gene,
            'gene_name': gene_name,
            'Sample_size': sample_count,  # Total number of samples (files)
            'amplification_count': count, # Count of significant amplification events for the gene across all samples
            'Amplification_sum (i.e. total log2(amplification CNV) across samples)': amp_sum, # Total log2 value of amplification CNV across samples
            'Amplification_sum_prevalence': amp_sum_prev,  # Average amplification (i.e. average log2(amplification CNV) across samples)
            'frequency_percentage': freq, # Percentage of samples with amplification for the gene
            'gistic_score': gistic,  # GISTIC-like score. i.e. calculated as amplification prevalance * frequency percentage of amplification
            'p_value': pval, # P-value from the binomial test for amplification
            'q_value': qval,  # Adjusted p-value for multiple testing (FDR)
            'significant': reject,  # Boolean indicating if the gene is significantly amplified based on the binomial test
            'empirical_p_value': pval_empirical, # Empirical p-value based on the null distribution
            'empirical_q_value': qval_empirical,  # Adjusted empirical p-value for multiple testing (FDR)
            'empirical_significant': empirical_reject # Boolean indicating if the gene is significantly amplified based on the empirical p-value
        })

    # Convert results to DataFrame for easier handling
    results = pd.DataFrame(results)
    results = results.sort_values(by='frequency_percentage', ascending=False)
    
    return results

In [ ]:
def gistic_scoring_mimic_deletion(files, file_dict_gene_dict, gene_id_translation_dict):
    """
    Mimics the GISTIC scoring for deletion events. This version will create a null based on the shuffled deletion values. 
    Also works to calculate the percentage of a chromosome the mutation covers
    
    Parameters:
    files (list): List of file paths to process.
    file_dict_gene_dict (dict): Dictionary mapping file names to gene dictionaries. contain the copy number values of each gene
    with <1 indicating high level deletion.

    
    Returns:
    dict: A dataframe with scores for each gene.
    1. 'gene': Gene identifier.
    2. 'gene_name': Human-readable gene name.
    3. 'Sample_size': Total number of samples (files) processed.
    4. 'deletion_count': Count of deletion events for the gene.
    5. 'deletion_sum (i.e. total log2(deletion CNV) across samples)': the sum of log2 values of deletion CNV across samples. 
        (-1 to adjust for diploid cells, so only deletion values and intensity are counted for)
    6. 'deletion_sum_prevalence (i.e. average log2(deletion CNV) across samples)': the average log2 value of deletion CNV across samples. (deletion_sum / sample_count)
    7. 'frequency_percentage': Percentage of samples with deletion for the gene.
    8. 'gistic_score': GISTIC-like score for the gene based on average deletion value.
    9. 'p_value': P-value from the binomial test for deletion .
    11. 'q_value': Adjusted p-value for multiple testing (FDR).
    14. 'significant': Boolean indicating if the gene is significantly deleted.
    12. 'empirical_p_value': Empirical p-value based on the null distribution.
    13. 'empirical_q_value': Adjusted empirical p-value for multiple testing (FDR).
    14. 'empirical_significant': Boolean indicating if the gene is significantly deleted based on the empirical p-value.
    15. 'start': Start position of the gene on the chromosome.
    16. 'end': End position of the gene on the chromosome.
    17. 'length': Length of the gene in base pairs (end - start + 1).
    18. 'chromosome': Chromosome on which the gene is located.
    19. 'coverage_percentage': Percentage of the chromosome that the gene covers.
    """
    
    ########################### creation of the calculations for the amplification effects #################################
    files = list(set(files)) # Ensure files is a list of unique file paths
    sample_count = len(files)
    
    # Step 1: Count amplification events and accumulate values
    ### Initialize dictionaries to hold amplification counts and values
    ### deletion_dict will hold the count of high level deletion events for each gene
    deletion_dict = {}
    del_values_sum = {}
    ## initialize deletion_dict with 0 for all genes
    all_genes = set()
    for f in files:
        all_genes.update(file_dict_gene_dict[f].keys())
    deletion_dict = {gene: 0 for gene in all_genes}
    del_values_sum = {gene: 0 for gene in all_genes}

    cnv_values = []
    for file in tqdm(files, desc = "Counting deletion events and accumulating values"):
        gene_dict = file_dict_gene_dict.get(file, {})
        for gene, value in gene_dict.items():
            ### skip if values is none
            ## check if the value is NaN, 'NA', 'nan', or not a number
            if pd.isnull(value) or value == 'NA' or value == 'nan' or not isinstance(value, (int, float, np.integer, np.floating)):
                cnv_values.append(2) # append a value of 2 to represent no deletion (i.e. normal diploid state) for the null distribution
                del_values_sum[gene] = del_values_sum.get(gene, 0) + 0
                continue
            cnv_values.append(value)
            # Mimicking GISTIC scoring for deletion
            if value == 1:
                ### accumulate the sum of values for each gene
                del_values_sum[gene] = del_values_sum.get(gene, 0) + -1 # -1 to account for a lesser deletion
            if value < 1:  # Assuming values <1 indicate high-level deletion
                deletion_dict[gene] = deletion_dict.get(gene, 0) + 1 ### accumulate the counts of deletion events for each gene
                del_values_sum[gene] = del_values_sum.get(gene, 0) + -2  # -2 to account for high-level deletion

    # Step 2: Binomial test per gene
    genes = list(deletion_dict.keys())
    ### essentiall the observed counts of deletion events for each gene
    counts = [deletion_dict[g] for g in genes]
    p_values = []

    ### Perform binomial test for each gene
    # Note: The binomial test checks if the observed count is significantly greater than expected under the null hypothesis
    # where the null hypothesis is that the probability of deletion is p_null.
    # The alternative hypothesis is that the observed count is significantly greater than expected.
    # This is similar to GISTIC's approach of assessing significance of deletion events.
    # So if the hypothesis is rejected, it indicates that the gene is significantly deleted across the samples.

    ### p_null is equal to background deletion rate across all samples
    p_null =np.sum(np.array(cnv_values) < 1) / len(cnv_values)

    for count in counts:
        result = binomtest(k=count, n=sample_count, p=p_null, alternative='greater')
        p_values.append(result.pvalue)

    ### Calculate a null distribution based on the shuffled deletion values
    # Create a null distribution by permuting the CNV values
    ### This will help in calculating empirical p-values
    ### Initialize a dictionary to hold the null distributions for each gene
    ### This will hold the counts of deletion events for each gene in the null distribution

    # shuffle the CNV values to create a null distribution
    np.random.seed(42)  # For reproducibility
    ### Initialize a dictionary to hold the null distributions for each gene
    null_distributions = {g: [] for g in genes}
    permutation_count = 1000  # Number of permutations to create a null distribution

    permuted_cnv_matrix = np.random.permutation(cnv_values)
    ### Create a quick permuted CNV matrix

    for gene in tqdm(genes, desc='Calculating null distributions'):
        ### This will help in calculating empirical p-values
        null_distributions[gene] = [np.sum(np.random.choice(permuted_cnv_matrix, size = sample_count) < 1) for _ in range(permutation_count)]

    ### calculate empirical p-values based on the null distribution
    p_empirical_values = []
    for gene in genes:
        observed_count = deletion_dict.get(gene, 0)
        null_counts = null_distributions[gene]
        # Calculate the empirical p-value as the proportion of null counts greater than or equal to the observed count
        ### works by counting how many times the null distribution has a count greater than or equal to the observed count
        ### The empirical p-value is calculated as the proportion of null counts that are greater than or equal to the observed count
        ### this is divided by the total number of permutations plus one to account for the observed count itself
        ### Thi gives the empirical p-value for the gene
        ### Note: This is a more efficient way to calculate the empirical p-value
        ### if the p_empirical is less than the p_null, then the gene is considered significantly amplified
        ### this is similar to the binomial test, but uses the null distribution to calculate the p-value
        ### This is because the null distribution is based on the shuffled CNV values, which is more representative of the data
        p_empirical = (np.sum(np.array(null_counts)>= observed_count)+1) / (permutation_count +1)
        p_empirical_values.append(p_empirical)

    # Step 3: Adjust p-values for multiple testing
    rejected, q_values, _, _ = multipletests(p_values, method='fdr_bh')
    # Adjusted q-values are the p-values adjusted for multiple testing
    rejected_empirical, q_values_empirical, _, _ = multipletests(p_empirical_values, method='fdr_bh')
    
    # Adjusted q-values are the p-values adjusted for multiple testing
    # Note: `rejected` indicates which hypotheses are rejected (i.e., significant)
    ### my own double check
    threshold = 0.05
    my_own_rejects = [True if q < threshold else False for q in q_values]
    my_own_rejects_empirical = [True if q < threshold else False for q in q_values_empirical]

    ### collect the prevalence of the genes high amplification across the entire cohort of samples
    frequency_percentage = [100 * deletion_dict.get(gene, 0) / sample_count for gene in genes]

    # Note: GISTIC scoring is typically more complex and involves additional steps like segmenting the data,
    # but here we mimic it by averaging the amplification values across samples.
    # This is a simplified version and may not capture all nuances of GISTIC scoring.
    # GISTIC scores are calculated as the average deletion value across samples for each gene.
    # attempt at gistic scoring
    gene_deletion_sum = [del_values_sum.get(gene, 0) for gene in genes] # This is the sum of values of CNV values across samples for each gene,
    # counted only deletions -1 for when ==1 and -2 for when <1
    # essentially only count deletions and not a negative value or reading due to amplification
    del_values_sum = {gene: abs(val) for gene, val in del_values_sum.items()}   # make all values positive for deletion sum and prevalence
    deletion_sum_prevalance = [(del_values_sum.get(gene, 0) / sample_count) for gene in genes]
    ## This is the average deletion value of CNV values across samples for each gene

    # This is a simplified version of GISTIC scoring
    # GISTIC scores are calculated as the average deletion value across samples multiplied by the frequency of deletion
    gistic_scores = list(np.array(deletion_sum_prevalance) * (np.array(frequency_percentage)/100))  # This is a simplified version of GISTIC scoring
    
    ## Translate gene IDs to gene names
    ## This will replace gene IDs with human-readable gene names using the translation dictionary
    gene_names = [gene_id_translation_dict.get(gene, "NA") for gene in genes]

    ### essentially the observed counts of deletion events for each gene
    # counts = [deletion_dict[g] for g in genes]

    # Step 4: Compile results
    results = []
    for gene, gene_name, count, del_sum_count, del_sum_prev, freq, gistic, pval, pval_empirical, qval, qval_empirical, reject, empirical_reject in zip(genes, gene_names, counts, gene_deletion_sum, deletion_sum_prevalance, 
                                                                                                                                                         frequency_percentage, gistic_scores, p_values, p_empirical_values, q_values, 
                                                                                                                                                         q_values_empirical, my_own_rejects, my_own_rejects_empirical):
        results.append({
            'gene': gene,
            'gene_name': gene_name,
            'Sample_size': sample_count,  # Total number of samples (files)
            'deletion_count': count,
            'Deletion_sum (i.e. total deletion CNV count across samples)': del_sum_count,  # Total deletion CNV count across samples
            'Deletion_sum_prevalence': del_sum_prev,  # Average deletion CNV count (i.e. average (deletion CNV count) across samples)'
            'frequency_percentage': freq, # Percentage of samples with significant deletion (<1) for the gene
            'gistic_score': gistic,  # GISTIC-like score. i.e. calculated as deletion prevalance * frequency percentage of deletion
            'p_value': pval, # P-value from the binomial test for deletion
            'q_value': qval, # Adjusted p-value for multiple testing (FDR)
            'significant': reject, # Boolean indicating if the gene is significantly deleted based on the binomial test
            'empirical_p_value': pval_empirical, # Empirical p-value based on the null distribution
            'empirical_q_value': qval_empirical, # Adjusted empirical p-value for multiple testing (FDR)
            'empirical_significant': empirical_reject, # Boolean indicating if the gene is significantly deleted based on the empirical p-value
        })

    # Convert results to DataFrame for easier handling
    results = pd.DataFrame(results)
    results = results.sort_values(by='frequency_percentage', ascending=False)
    return results

### HPV+

#### Amplification

In [ ]:
hpv_pos_amplification_results = gistic_scoring_mimic_amplification(HPV_positive_files, file_dict_gene_dict, gene_id_translation_dict)

In [ ]:
hpv_pos_amplification_results

In [ ]:
print(len(set(hpv_pos_amplification_results['gene_name'])))

In [ ]:
top_hpv_pos_amplification_results = hpv_pos_amplification_results[hpv_pos_amplification_results['q_value']<0.05][hpv_pos_amplification_results['empirical_q_value'] < 0.05]

In [ ]:
print(len(set(top_hpv_pos_amplification_results['gene_name'])))

In [ ]:
hpv_pos_amplification_results.to_csv('Results/CNV results/HPV positive amplification genes.csv', index=False)

#### Deletion

In [ ]:
hpv_pos_deletion_results = gistic_scoring_mimic_deletion(HPV_positive_files, file_dict_gene_dict, gene_id_translation_dict)

In [ ]:
hpv_pos_deletion_results.sort_values(by = 'frequency_percentage', ascending=False)

In [ ]:
hpv_pos_deletion_results.to_csv('Results/CNV results/HPV positive deletion genes.csv', index=False)

In [ ]:
hpv_pos_deletion_results

In [ ]:
print(len(set(hpv_pos_deletion_results['gene_name'])))

In [ ]:
top_hpv_pos_deletion_results = hpv_pos_deletion_results[hpv_pos_deletion_results['q_value']<0.05][hpv_pos_deletion_results['empirical_q_value'] < 0.05]

In [ ]:
print(len(set(top_hpv_pos_deletion_results['gene_name'])))

### HPV-

#### Amplification

In [ ]:
hpv_neg_amplification_results = gistic_scoring_mimic_amplification(HPV_negative_files, file_dict_gene_dict, gene_id_translation_dict)

In [ ]:
hpv_neg_amplification_results

In [ ]:
len(set(hpv_neg_amplification_results['gene_name']))

In [ ]:
top_hpv_neg_amplification_results = hpv_neg_amplification_results[hpv_neg_amplification_results['q_value']<0.05][hpv_neg_amplification_results['empirical_q_value'] < 0.05]

In [ ]:
len(set(top_hpv_neg_amplification_results['gene_name']))

In [ ]:
hpv_neg_amplification_results.to_csv('Results/CNV results/HPV negative amplification genes.csv', index=False)

#### Deletion

In [ ]:
len(HPV_negative_files)

In [ ]:
hpv_neg_deletion_results = gistic_scoring_mimic_deletion(HPV_negative_files, file_dict_gene_dict, gene_id_translation_dict)

In [ ]:
hpv_neg_deletion_results

In [ ]:
print(len(set(hpv_neg_deletion_results['gene_name'])))

In [ ]:
top_hpv_neg_deletion_results = hpv_neg_deletion_results[hpv_neg_deletion_results['q_value']<0.05][hpv_neg_deletion_results['empirical_q_value'] < 0.05]

In [ ]:
print(len(set(top_hpv_neg_deletion_results['gene_name'])))

In [ ]:
hpv_neg_deletion_results.to_csv('Results/CNV results/HPV negative deletion genes.csv', index=False)